In [9]:
import os
import requests
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from time import sleep

In [10]:
USER_AGENT = {"User-Agent": "Pikechu"}

In [ ]:
#latitude = '33.211952'
latitude = '37.4277'
#longitude = '-87.545920'
longitude = '-122.1701'

url = f'https://api.weather.gov/points/{latitude},{longitude}'
response = requests.get(url, headers=USER_AGENT).json()

# Display response from API
response

In [12]:
office = response["properties"]["gridId"]
gridX = response["properties"]["gridX"]
gridY = response["properties"]["gridY"]

# Print grid location data
print(f'Office code: {office}, Grid coordinates: {gridX}, {gridY}')

Office code: MTR, Grid coordinates: 90, 88


In [5]:
url = f'https://api.weather.gov/gridpoints/{office}/{gridX},{gridY}/forecast/hourly'
response = requests.get(url, headers=USER_AGENT).json()

# Display first result
response["properties"]["periods"][0]

{'number': 1,
 'name': '',
 'startTime': '2026-01-05T14:00:00-08:00',
 'endTime': '2026-01-05T15:00:00-08:00',
 'isDaytime': True,
 'temperature': 56,
 'temperatureUnit': 'F',
 'temperatureTrend': None,
 'probabilityOfPrecipitation': {'unitCode': 'wmoUnit:percent', 'value': 65},
 'dewpoint': {'unitCode': 'wmoUnit:degC', 'value': 10},
 'relativeHumidity': {'unitCode': 'wmoUnit:percent', 'value': 80},
 'windSpeed': '15 mph',
 'windDirection': 'SSE',
 'icon': 'https://api.weather.gov/icons/land/day/rain_showers,70?size=small',
 'shortForecast': 'Rain Showers Likely',
 'detailedForecast': ''}

In [6]:
url = f"https://api.weather.gov/gridpoints/{office}/{gridX},{gridY}"
data = requests.get(url, headers=USER_AGENT).json()

In [ ]:
data["properties"]

In [ ]:
import xarray as xr
import s3fs

fs = s3fs.S3FileSystem(anon=True)



path = (
  "noaa-hrrr-bdp-pds/hrrr.20260105/conus/"
  "hrrr.t00z.wrfsfcf01.grib2"
)



with fs.open(path, 'rb') as f:
    ds = xr.open_dataset(
        f,
        engine="cfgrib",
        backend_kwargs={"filter_by_keys": {"typeOfLevel": "surface"}}
    )

ds["vis"]


In [ ]:
import s3fs
import xarray as xr

fs = s3fs.S3FileSystem(anon=True)

key = "hrrr.20260105/conus/hrrr.t00z.wrfsfcf01.grib2"
path = f"noaa-hrrr-bdp-pds/{key}"

with fs.open(path, "rb") as f:
    ds = xr.open_dataset(
        f,
        engine="cfgrib",
        backend_kwargs={"filter_by_keys": {"typeOfLevel": "surface"}}
    )

ds


In [3]:
import xarray as xr

path = "hrrr.t00z.wrfsfcf01.grib2"

ds_sfc_inst = xr.open_dataset(
    path,
    engine="cfgrib",
    backend_kwargs={
        "filter_by_keys": {
            "typeOfLevel": "surface",
            "stepType": "instant",
        },
        "indexpath": ""   # 建议加上，避免 .idx 缓存导致反复报错
    },
)

list(ds_sfc_inst.data_vars)[:40], ds_sfc_inst


(['vis',
  'gust',
  'sp',
  'orog',
  't',
  'cnwat',
  'sdwe',
  'snowc',
  'sde',
  'cpofp',
  'prate',
  'csnow',
  'cicep',
  'cfrzr',
  'crain',
  'fsr',
  'fricv',
  'ishf',
  'slhtf',
  'veg',
  'unknown',
  'lai',
  'gflux',
  'vgtyp',
  'cape',
  'cin',
  'sdswrf',
  'sdlwrf',
  'suswrf',
  'sulwrf',
  'cfnsf',
  'vbdsf',
  'vddsf',
  'blh',
  'lsm',
  'siconc'],
 <xarray.Dataset> Size: 305MB
 Dimensions:     (y: 1059, x: 1799)
 Coordinates:
     time        datetime64[ns] 8B ...
     step        timedelta64[ns] 8B ...
     surface     float64 8B ...
     latitude    (y, x) float64 15MB ...
     longitude   (y, x) float64 15MB ...
     valid_time  datetime64[ns] 8B ...
 Dimensions without coordinates: y, x
 Data variables: (12/36)
     vis         (y, x) float32 8MB ...
     gust        (y, x) float32 8MB ...
     sp          (y, x) float32 8MB ...
     orog        (y, x) float32 8MB ...
     t           (y, x) float32 8MB ...
     cnwat       (y, x) float32 8MB ...
     ... 

In [6]:
import numpy as np

lat0, lon0 = 37.4277, -122.1701


# 兼容变量名
lat_name = "latitude" if "latitude" in ds_sfc_inst else "lat"
lon_name = "longitude" if "longitude" in ds_sfc_inst else "lon"

lat2d = ds_sfc_inst[lat_name].values
lon2d = ds_sfc_inst[lon_name].values

# 经度统一到 [-180, 180]，避免 0-360/±180 混用
lon2d = ((lon2d + 180) % 360) - 180
lon0n = ((lon0 + 180) % 360) - 180

# 简单距离（足够用于最近格点）
dist2 = (lat2d - lat0)**2 + (lon2d - lon0n)**2
iy, ix = np.unravel_index(np.argmin(dist2), dist2.shape)

vis_m = ds_sfc_inst["vis"].isel(y=iy, x=ix).item()
vis_km = vis_m / 1000
(iy, ix, vis_m, vis_km)


(586, 182, 17100.0, 17.1)

In [8]:
ds_sfc_inst["time"].values

numpy.datetime64('2026-01-05T00:00:00.000000000')